In [ ]:
#| default_exp api

# API

> WeCom access token management, message sending, and Markdown stripping.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re, time
import httpx
from solveit_wxbot.config import CORP_ID, SECRET, AGENT_ID, CHUNK_SIZE, TOKEN_BUFFER

## Markdown 清理

WeCom 客户端不渲染 Markdown，发送前需将 AI 回复中的代码块、加粗、标题、列表、链接等语法替换为纯文本或简单符号。

In [ ]:
#| export
def strip_md(
    s: str  # Text potentially containing Markdown syntax
) -> str:
    "Strip common Markdown so text reads cleanly in WeChat."
    s = re.sub(r'```[\s\S]*?```', '', s)
    for pat, repl in [
        (r'`([^`]+)`',              r'\1'),  # inline code
        (r'\*+(.+?)\*+',            r'\1'),  # bold / italic
        (r'^#{1,6}\s+',             ''   ),  # headings
        (r'^\s*[-*+]\s+',           '• ' ),  # bullets
        (r'\[([^\]]+)\]\([^\)]+\)', r'\1'),  # links
    ]:
        s = re.sub(pat, repl, s, flags=re.MULTILINE)
    return s.strip()

## Access Token 管理

WeCom Access Token 有效期约2小时，通过模块级缓存 `_token_cache` 避免频繁请求。`get_access_token` 在到期前 `TOKEN_BUFFER` 秒自动刷新。

In [ ]:
#| export
_token_cache = {'token': '', 'expires': 0}

async def get_access_token() -> str:
    "Return a valid WeCom access token, refreshing from the API 5 min before expiry."
    now = time.time()
    if _token_cache['token'] and now < _token_cache['expires']:
        return _token_cache['token']
    async with httpx.AsyncClient() as c:
        r = await c.get('https://qyapi.weixin.qq.com/cgi-bin/gettoken',
                        params={'corpid': CORP_ID, 'corpsecret': SECRET})
    d = r.json()
    if d.get('errcode', 0) != 0: raise Exception(f"获取 token 失败: {d}")
    _token_cache.update(token=d['access_token'], expires=now + d['expires_in'] - TOKEN_BUFFER)
    return _token_cache['token']

## 消息发送

WeCom 单条消息有字符限制，`send_text` 按 `CHUNK_SIZE` 自动分段发送，并在每段失败时记录日志而不中断后续分段。

In [ ]:
#| export
async def send_text(
    user_id: str,  # WeCom recipient user ID
    text: str      # Message text; Markdown is stripped automatically
):
    "Send `text` via the WeCom API, chunking into `CHUNK_SIZE`-char segments."
    token = await get_access_token()
    url = f'https://qyapi.weixin.qq.com/cgi-bin/message/send?access_token={token}'
    text = strip_md(text)
    async with httpx.AsyncClient() as c:
        for i in range(0, len(text), CHUNK_SIZE):
            r = await c.post(url, json={
                'touser': user_id, 'msgtype': 'text',
                'agentid': int(AGENT_ID), 'text': {'content': text[i:i+CHUNK_SIZE]}})
            if (d := r.json()).get('errcode', 0) != 0:
                print(f"⚠️ 发送失败: {d}")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()